# Step 9A · CTRL-001: Standalone ViT-only control (seed 42)

**Goal:** Before inventing a new CNN–ViT method, test whether a genuine standalone ViT can equal or exceed our previously observed *CNN-collapsed independent reconstruction* on **exactly the same 35,005 original training-derived validation windows**.

**This is an engineering control, not a novelty claim or published baseline outperformance.**

## What is locked

- **Dataset:** Original `dsshayan/unsw-nb15/UNSW_NB15_training-set.csv` (175,341 rows). Check SHA-256. **Official testing file is not read or attached.**
- **Train/validation:** Original saved `split_indices.npz`: same original 140,332 training rows and 35,009 validation rows. Same 140,328/35,005 constructed five-row windows. SHA-256 checked against existing research archive.
- **Preprocessing:** Replay the ORIGINAL train-only feature-index vocabularies and MinMax parameters from `preprocessing.json`, then identically serialize five-row traffic images.
- **Model:** Reuse the original four-layer, 128-dimensional, eight-head ViT encoder; mean-pool tokens and classify using 128→128→10 with GELU/dropout. There is **no CNN or crossattention**. That architecture difference is explicit and versioned.
- **Training:** Same 50 epochs, batch 64, focal gamma=2, AdamW learning rate .001, weight decay .001, seed 42, and best-checkpoint selection by validation macro-F1.
- **Results:** Actual probabilities, metrics, all 10-class confusion counts, 14 × PNG/SVG/PDF figures, numeric source CSVs, exact code/configuration/SHA references.
- **Preservation:** All checkpoints and original per-row predictions saved privately in a checksum-verified **draft GitHub Release**, not the public repository; public GitHub gets source code, aggregate metrics and SVG/PNG/PDF figures.

**Important controls:** We will not assume five-row CSV adjacency means chronological traffic. We will add window-size-one and row-permutation controls later. This is seed 42 only; it does not establish statistical superiority.

### Kaggle settings
GPU enabled, Internet enabled, attach only `dsshayan/unsw-nb15` as input, authorize existing secret `pushRare`. Do not print or paste the secret. Notebook stages are split intentionally: late figure/report errors cannot force 50 training epochs to repeat.

In [ ]:
# Cell 1 — Clone and restore original SCV-IDS split/preprocessor from verified archive.
from pathlib import Path
import os,subprocess,sys

REPO=Path("/kaggle/working/rare-attack-ids-ctrl001")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"CTRL-VIT-001-S42"
WEIGHTS=BASE/"checkpoints_ctrl_vit_001_s42"
ARCHIVE=Path("/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
SOURCE=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
URL="https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"
assert SOURCE.is_file(), "Attach the dsshayan/unsw-nb15 dataset"
if REPO.exists():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",URL,str(REPO)],check=True)
print("Research code SHA:",subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())
import torch
assert torch.cuda.is_available(),"Enable a Kaggle T4/P100 GPU accelerator"
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                "--format=csv,noheader"],check=True)

env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["OPENBLAS_NUM_THREADS"]="1"
env["MKL_NUM_THREADS"]="1"
env["OMP_NUM_THREADS"]="1"
env["MPLBACKEND"]="Agg"
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret("pushRare")
secure_env=env.copy()
secure_env["GITHUB_TOKEN"]=token
try:
    subprocess.run([
        sys.executable,str(REPO/"scripts"/"restore_scv_release.py"),
        "--base",str(BASE),"--archive",str(ARCHIVE)
    ],cwd=str(REPO),env=secure_env,check=True)
finally:
    secure_env.pop("GITHUB_TOKEN",None)
    del token
print("Original archived split and fitted preprocessor available.")

In [ ]:
# Cell 2 — The ONLY training cell. Checkpoint after EVERY epoch; resume supported.
CONFIG=REPO/"configs"/"ctrl001_vit_only_seed42.json"
subprocess.run([
    sys.executable,str(REPO/"scripts"/"train_ctrl001_vit.py"),
    "--repo",str(REPO),"--config",str(CONFIG),
    "--output",str(RUN),"--checkpoints",str(WEIGHTS)
],cwd=str(REPO),env=env,check=True)
print("50-epoch standalone ViT control complete. Keep this Kaggle session.")

In [ ]:
# Cell 3 — NO TRAINING: derive final metrics/CSV and 14 × 3 publication figures.
subprocess.run([
    sys.executable,str(REPO/"scripts"/"finalize_ctrl001_vit.py"),
    "--run",str(RUN),"--checkpoints",str(WEIGHTS),
    "--reference",str(BASE/"SCV-IDS-REPRO-001")
],cwd=str(REPO),env=env,check=True)
print((RUN/"run_report.md").read_text(encoding="utf-8"))

In [ ]:
# Cell 4 — First preserve complete model in draft GitHub Release;
# then sync compact reproducible figure/report sources to public GitHub.
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret("pushRare")
secure_env=env.copy()
secure_env["GITHUB_TOKEN"]=token
new_archive=Path("/kaggle/working/CTRL-VIT-001-S42_FULL_ARTIFACTS.tar")
try:
    subprocess.run([
        sys.executable,str(REPO/"scripts"/"archive_ctrl001_vit.py"),
        "--run",str(RUN),"--checkpoints",str(WEIGHTS),
        "--archive",str(new_archive)
    ],cwd=str(REPO),env=secure_env,check=True)
    subprocess.run([
        sys.executable,str(REPO/"scripts"/"sync_ctrl001_vit.py"),
        "--run",str(RUN)
    ],cwd=str(REPO),env=secure_env,check=True)
finally:
    secure_env.pop("GITHUB_TOKEN",None)
    del token
print("All experiment artifacts preserved. Verify the draft archive and commit URLs.")

## Return to ChatGPT

Send **`VERIFIED CTRL-VIT SYNC COMMIT: ...`** and **`CTRL-VIT PRIVATE RELEASE VERIFIED: ...`**.

Do not run the *official UNSW test* here, and do not compare different evaluation subsets. The original full-validation macro-F1 was 55.8664% for the best reconstructed SCV checkpoint, with 82.7396% accuracy. The new ViT control is tested on *identical* label targets, enabling a fairer baseline engineering comparison, though only one seed.

**If any finalization figure fails after 50 epochs, rerun only Cell 3. Do not delete the checkpoint folder or repeat Cell 2.**

After successful versioning, we can build the **healthy CNN–ViT corrected architecture** (V2), test activation health, and later run the row permutation/window=1 and multi-seed controls.